# Lab 4 — LangChain (EP02)

In Labs 1–3 we wrote everything ourselves: building `messages`, sending `response_format`, `model_validate_json`, the retry loop.
In this lab we rebuild **the same email sorter** with **LangChain**, and match it piece by piece with the code we wrote

| We wrote it (Labs 1–2) | LangChain |
|---|---|
| `client` + `MODEL` + `EXTRA` | `ChatOpenAI(...)` |
| `FEW_SHOT` (a list of messages) | `ChatPromptTemplate` + `FewShotChatMessagePromptTemplate` |
| `schema_of()` + `response_format` + `model_validate_json` | `llm.with_structured_output(StudentEmail)` |
| `for email in EMAILS:` | `chain.batch([...])` (runs in parallel) |
| `extract(email, tries=3)` | `.with_retry(stop_after_attempt=3)` |

Versions: `langchain-openai==1.7.2`, `langchain-core==1.7.0` (the same set as EP05 Lab 5)

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.13.5, §19.3 — CC BY-SA 4.0

## 0. Set up before you start

Same as Lab 1: Python 3.10+ or Colab, and `OPENROUTER_API_KEY` in Colab Secrets or a `.env` file
(how to set up the key, and why we chose this model and provider, are in Lab 1)

In [ ]:
%pip install -q "langchain-openai==1.7.2" "langchain-core==1.7.0" "pydantic>=2.7" python-dotenv

In [ ]:
import os, re
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, field_validator

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

EMAILS = [
    "Dear lecturer, I have the flu. May I hand in homework 2 for AI301 three days late? I've attached a medical certificate.",
    "Which chapters does the midterm exam cover?",
    "The LMS crashed when I submitted my CS214 work, and I lost marks for being late. I don't think that's fair.",
    "Can we move the group report deadline to Monday? Our members have an exam in another course.",
    "Do we need our own API key for Lab 3 of AI301?",
    "I've submitted my homework twice but my score still shows zero. Could you check, please?",
]

class StudentEmail(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intent: Literal["question", "extension", "complaint"] = Field(
        description="question = asks for information · extension = asks to hand in work late · complaint = reports a problem or is unhappy")
    urgency: Literal["low", "medium", "high"] = Field(description="high = a deadline or marks are at stake")
    course: str | None = Field(description="the course code written in the email, e.g. AI301; null if there is none")
    summary: str = Field(description="a one-sentence summary")

    @field_validator("course")
    @classmethod
    def must_be_code(cls, v):
        if v is not None and not re.fullmatch(r"[A-Z]{2,3}\d{3}", v):
            raise ValueError(f"not a course code: {v!r}")
        return v

## 1. `ChatOpenAI` = client + MODEL + EXTRA

In [ ]:
from langchain_openai import ChatOpenAI

EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["parasail/bf16"], "allow_fallbacks": False}}
llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it", base_url="https://openrouter.ai/api/v1",
                 api_key=load_key(), extra_body=EXTRA)
print(llm.invoke("Greet the students in one short sentence").content)

## 2. Few-shot with a prompt template (Lab 1)

`ChatPromptTemplate` = a job sheet with blanks to fill (`{email}`) · the examples are data, not hand-built messages

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

EXAMPLES = [{"email": "Can I hand in my work late? Something came up at home.", "label": "extension"},
            {"email": "Which chapters are on next week's quiz?", "label": "question"},
            {"email": "My exam score disappeared from the system and nobody has replied.", "label": "complaint"}]
few_shot = FewShotChatMessagePromptTemplate(
    examples=EXAMPLES,
    example_prompt=ChatPromptTemplate.from_messages([("human", "{email}"), ("ai", "{label}")]))

In [ ]:
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the student email as one of: question / extension / complaint"),
    few_shot,
    ("human", "{email}")])
classify = classify_prompt | llm | StrOutputParser()

In [ ]:
labels = classify.batch([{"email": e} for e in EMAILS])
print(labels)

## 3. Structured output with `with_structured_output` (Lab 2)

One line replaces `schema_of()` + `response_format` + `model_validate_json`

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a teaching assistant. Read the student email in <email> and fill in the schema using only information in the email"),
    ("human", "<email>\n{email}\n</email>")])
structured = llm.with_structured_output(StudentEmail, method="json_schema", strict=True)
extractor = email_prompt | structured

LABEL = {"question": "Question", "extension": "Extension", "complaint": "Complaint"}
for info in extractor.batch([{"email": e} for e in EMAILS]):
    print(LABEL[info.intent], info.urgency, repr(info.course), "|", info.summary)

## 4. `.with_retry()` instead of `extract(email, tries=3)`, and what it doesn't do

`with_retry` **resends the same input** when an exception is raised; it doesn't tell the model what was wrong.
We try it with a plain prompt (no system prompt and no `<email>` tags), which in Lab 2 often got course wrong

In [ ]:
plain_prompt = ChatPromptTemplate.from_messages([("human", "Read the student email and extract the information\nEmail: {email}")])
retrying = (plain_prompt | structured).with_retry(stop_after_attempt=3)
for e, out in zip(EMAILS, retrying.batch([{"email": e} for e in EMAILS], return_exceptions=True)):
    print(e[:30], "→", type(out).__name__ if isinstance(out, Exception) else repr(out.course))

## Summary: our own code vs LangChain

| Task | Our own code (Labs 1–2) | LangChain (this lab) |
|---|---|---|
| call the model | `client` + `MODEL` + `EXTRA` + `ask()` | `ChatOpenAI(...)` |
| few-shot | a list of messages built by hand | `FewShotChatMessagePromptTemplate` |
| joining steps | nested function calls | `prompt \| llm \| parser` |
| structured output | `schema_of()` + `response_format` + `model_validate_json` | `with_structured_output(StudentEmail)` |
| many emails | a `for` loop | `.batch()` (in parallel) |
| asking again | `extract(tries=3)` | `.with_retry(stop_after_attempt=3)` |

**What LangChain doesn't do for us**
- it doesn't write a good prompt or schema: `StudentEmail`, the field descriptions and the validator are still ours
- `with_retry` resends the same input and **doesn't tell the model what was wrong** (libraries such as Instructor send the error back for the model to fix)
- it doesn't check for hallucination: the quote check in Lab 3 is still our code

**What we gain:** one chain can be called with `.invoke` (one email) and `.batch` (many in parallel),
and a chain like this is what becomes a node in LangGraph in EP05

**Exercise:** write `extract_with_feedback(email)` that, when validation fails, adds the error message to the next prompt
("last time course was wrong: ...") and compare it with `with_retry` on `plain_prompt`

▶ **Next: EP02 recap**